# Critical Values: the Decision Rule You Can Write in Advance

The p-value method needs the data before it can speak. Flip the question — *what is the least extreme sample that would still reject $H_0$?* — and you get the **critical value** $k_\alpha$: the observation whose p-value is exactly $\alpha$. Anything more extreme automatically has $p \le \alpha$, so "reject when the observed statistic passes $k_\alpha$" is a complete decision rule — and one you can compute **before seeing any data**, from the design conditions alone. For the height test at $\alpha = 0.05$: $k_{0.05} = 68.26$, so the observed 68.442 rejects; tighten to $\alpha = 0.01$ and $k_{0.01} = 68.91$ — now it doesn't. Same conclusions as the p-value method, always — plus a new superpower: an advance rule makes Type II error probabilities computable. It is based on the fifth video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

Notebook 04 kept meeting one special observation — the borderline sample where $p = \alpha$ exactly. This notebook names it, computes it for all three tails, and shows why pre-registering the threshold (rather than computing p after the fact) is more than a convenience.

After completing this notebook you will be able to:

- Define the critical value $k_\alpha$ and read the subscript: different $\alpha$'s, different thresholds.
- Compute critical values as quantiles of the null distribution — $1-\alpha$ (right), $\alpha$ (left), and the $\alpha/2$ pair (two-tailed) — and state each decision rule.
- Reproduce the video's numbers: $k_{0.05} = 68.26$ (reject) and $k_{0.01} = 68.91$ (don't).
- Verify that the critical-value method and the p-value method always agree.
- Use the advance rule to do what p-values alone can't: compute a Type II error probability.

## 2. Theory

### The least extreme rejecting sample

The p-value rule rejects when $p < \alpha$. Ask: what's the *mildest* observation that still clears the bar? It's the one with $p = \alpha$ exactly — the **critical value**, written $k_\alpha$ to stress that it depends on your chosen $\alpha$. Its useful property: **any observed statistic more extreme than $k_\alpha$ has a p-value of $\alpha$ or less.** So the whole test compresses into a threshold comparison.

### The height example, thresholded

Right-tailed test, $H_0: \mu = 66.7$ vs $H_1: \mu > 66.7$, with $n = 10$, $\sigma = 3$, so under $H_0$: $\bar{X} \sim N(66.7,\, (3/\sqrt{10})^2)$.

- **$\alpha = 0.05$:** $k_{0.05}$ leaves area 0.05 to its right — i.e. it is the $1 - 0.05$ **quantile** of the null distribution:
$$
k_{0.05} = 66.7 + z_{0.95}\cdot\tfrac{3}{\sqrt{10}} = 68.26
\qquad\Rightarrow\qquad \text{reject } H_0 \text{ if } \bar{x} > 68.26
$$
The observed 68.442 exceeds it: reject — exactly the p-value method's conclusion.
- **$\alpha = 0.01$:** a smaller $\alpha$ pushes the threshold right — more evidence demanded:
$$
k_{0.01} = 68.91 \qquad\Rightarrow\qquad 68.442 \not> 68.91:\;\text{cannot reject at } 0.01
$$

### Critical values for every tail

| test | critical value(s) | decision rule |
|---|---|---|
| right-tailed | $k_\alpha$ = quantile $1-\alpha$ of the null distribution | reject if $t_{obs} > k_\alpha$ |
| left-tailed | $k_\alpha$ = quantile $\alpha$ | reject if $t_{obs} < k_\alpha$ |
| two-tailed | $k_{\alpha,1}$ = quantile $1-\alpha/2$ and $k_{\alpha,2}$ = quantile $\alpha/2$ | reject if $t_{obs} > k_{\alpha,1}$ or $t_{obs} < k_{\alpha,2}$ |

(Two-tailed: the error budget splits between the tails, $\alpha/2$ each, hence *two* critical values.)

### Why bother, if p-values already work?

1. **You can define the rule in advance** — no sample needed, only the design conditions: sample size, the null value, and whatever is known about the population's distribution.
2. **The two methods always agree.** $t_{obs}$ beyond $k_\alpha$ $\iff$ $p < \alpha$; they are the same comparison read from different axes.
3. **Advance rules unlock Type II analysis.** With a fixed, data-independent decision rule ("reject iff $\bar{x} > 68.26$"), the probability of *failing* to reject when some specific alternative is true becomes a straightforward computation — the subject of the next video.

## 3. Worked Examples

Throughout: SE $= 3/\sqrt{10} = 0.949$.

### Example 1: the video's two thresholds

$k_{0.05} = 66.7 + 1.645 \times 0.949 = 68.26$; $\;k_{0.01} = 66.7 + 2.326 \times 0.949 = 68.91$. The observed 68.442 sits *between* them — which is just another way of saying its p-value (0.0332) sits between 0.01 and 0.05.

### Example 2: the other two tails, same design

- Left-tailed, $\alpha = 0.05$: $k_{0.05} = 66.7 - 1.645 \times 0.949 = 65.14$; reject if $\bar{x} < 65.14$. (Notebook 04's decreased-heights sample, $\bar{x} = 64.252$, clears it: reject — matching its p of 0.0049.)
- Two-tailed, $\alpha = 0.05$: quantiles $0.025$ and $0.975$ give $66.7 \pm 1.96 \times 0.949 = [64.84,\; 68.56]$; reject *outside* the pair. The observed 68.442 is inside: don't reject — matching its two-tailed p of 0.0663.

### Example 3: the equivalence, stated once and for all

The p-value of an observation and its position relative to $k_\alpha$ are monotone images of each other: $\bar{x} > k_\alpha \iff P(\bar{X} > \bar{x} \mid H_0) < \alpha$. One rule, two dashboards — and a bonus reading: **the p-value is the smallest $\alpha$ at which your observation would still reject** (Exercise 4 makes this precise).

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(55)

MU0, SIGMA, N = 66.7, 3.0, 10
SE = SIGMA / np.sqrt(N)

def z_quantile(p):
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

Phi = lambda z: 0.5 * (1 + float(sp.erf(sp.Float(z) / sp.sqrt(2))))

def critical_values(alpha, tail):
    """Critical value(s) of the mean test under H0: mu = MU0 (sigma known)."""
    if tail == "right":
        return MU0 + z_quantile(1 - alpha) * SE
    if tail == "left":
        return MU0 + z_quantile(alpha) * SE
    if tail == "two":
        half = z_quantile(1 - alpha / 2) * SE
        return MU0 - half, MU0 + half

### The video's thresholds, and the verdicts they deliver

In [ ]:
x_obs = 68.442
for alpha in (0.05, 0.01):
    k = critical_values(alpha, "right")
    verdict = "REJECT" if x_obs > k else "cannot reject"
    print(f"alpha = {alpha}: k_{alpha} = {k:.2f}  ->  x-bar = 68.442 "
          f"{'>' if x_obs > k else '<'} k  ->  {verdict}")

print(f"\nleft-tailed  alpha = 0.05: k = {critical_values(0.05, 'left'):.2f} "
      f"(64.252 rejects: {64.252 < critical_values(0.05, 'left')})")
lo, hi = critical_values(0.05, "two")
print(f"two-tailed   alpha = 0.05: reject outside [{lo:.2f}, {hi:.2f}] "
      f"(68.442 inside -> don't reject)")

### The two methods agree — everywhere

Sweep every possible observation and take both decisions: by p-value and by critical value. The decision vectors are identical, observation for observation.

In [ ]:
alpha = 0.05
xbars = np.linspace(64, 71, 2001)
p_values = np.array([1 - Phi((xb - MU0) / SE) for xb in xbars])

decision_by_p = p_values < alpha
decision_by_k = xbars > critical_values(alpha, "right")

print(f"observations checked: {len(xbars)}")
print(f"decisions identical:  {(decision_by_p == decision_by_k).all()}")
print(f"first rejecting observation: {xbars[decision_by_k.argmax()]:.3f} "
      f"(the critical value: {critical_values(alpha, 'right'):.3f})")

## 5. Visualizations

### Two $\alpha$'s, two thresholds

Smaller $\alpha$ slides $k_\alpha$ deeper into the tail — the observed 68.442 clears the 0.05 bar but not the 0.01 bar.

In [ ]:
x_axis = np.linspace(MU0 - 4 * SE, MU0 + 4 * SE, 700)
density = np.exp(-0.5 * ((x_axis - MU0) / SE) ** 2) / (SE * np.sqrt(2 * np.pi))

plt.figure(figsize=(10, 4.8))
plt.plot(x_axis, density, color="black", linewidth=1.6)
for alpha, color in [(0.05, "tab:orange"), (0.01, "tab:red")]:
    k = critical_values(alpha, "right")
    plt.fill_between(x_axis, density, where=x_axis >= k, color=color, alpha=0.45,
                     label=f"$\\alpha$ = {alpha}: reject beyond $k$ = {k:.2f}")
    plt.axvline(k, color=color, linestyle="--", linewidth=1.4)
plt.axvline(x_obs, color="tab:blue", linewidth=2.2,
            label="observed 68.442")
plt.axvline(MU0, color="black", linestyle=":", linewidth=1)
plt.xlabel(r"$\bar{x}$ (inches)")
plt.ylabel("density under $H_0$")
plt.title("Stricter alpha, farther threshold: 68.442 rejects at 0.05 but not at 0.01")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The critical values of all three tails

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2), sharey=True)
alpha = 0.05
k_r = critical_values(alpha, "right")
k_l = critical_values(alpha, "left")
k_2lo, k_2hi = critical_values(alpha, "two")
panels = [
    (f"right-tailed\nreject if $\\bar{{x}} > {k_r:.2f}$", x_axis >= k_r, [k_r]),
    (f"left-tailed\nreject if $\\bar{{x}} < {k_l:.2f}$", x_axis <= k_l, [k_l]),
    (f"two-tailed\nreject outside [{k_2lo:.2f}, {k_2hi:.2f}]",
     (x_axis <= k_2lo) | (x_axis >= k_2hi), [k_2lo, k_2hi]),
]
for ax, (title, region, ks) in zip(axes, panels):
    ax.plot(x_axis, density, color="black", linewidth=1.6)
    ax.fill_between(x_axis, density, where=region, color="tab:red", alpha=0.5)
    for k in ks:
        ax.axvline(k, color="tab:red", linestyle="--", linewidth=1.4)
    ax.axvline(MU0, color="black", linestyle=":", linewidth=1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$\bar{x}$ (inches)")
axes[0].set_ylabel("density under $H_0$")
plt.suptitle(r"Critical values are quantiles of the null distribution ($\alpha$ = 0.05)")
plt.tight_layout()
plt.show()

### One rule, two dashboards

The p-value axis and the observation axis tell the same story: $p$ dips below $\alpha$ at exactly the observation where $\bar{x}$ crosses $k_\alpha$.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9.5, 4.6))
ax1.plot(xbars, p_values, color="tab:blue", linewidth=2, label="p-value of the observation")
ax1.axhline(0.05, color="tab:blue", linestyle=":", linewidth=1.2)
ax1.set_xlabel(r"observed sample mean $\bar{x}$")
ax1.set_ylabel("right-tailed p-value", color="tab:blue")
ax1.set_yscale("log")
k = critical_values(0.05, "right")
ax1.axvline(k, color="tab:red", linestyle="--", linewidth=1.5,
            label=f"critical value $k_{{0.05}}$ = {k:.2f}")
ax1.fill_betweenx([1e-5, 1], k, xbars[-1], color="tab:red", alpha=0.10)
ax1.annotate("reject zone\n($p < \\alpha$ and $\\bar{x} > k$, always together)",
             xy=(69.6, 0.3), ha="center", fontsize=9, color="tab:red")
ax1.set_ylim(1e-5, 1)
ax1.grid(True, alpha=0.3)
ax1.legend(loc="lower left", fontsize=9)
ax1.set_title("p < alpha exactly where the observation passes the critical value")
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Compute $k_{0.10}$ for the right-tailed height test and the verdict on $\bar{x} = 68.442$. Then order $k_{0.10}, k_{0.05}, k_{0.01}$ and state the pattern.

**Exercise 2.** For the left-tailed test at $\alpha = 0.01$: critical value, decision rule, and the verdict on notebook 04's $\bar{x} = 64.252$. Check consistency with its p-value 0.0049.

### Intermediate

**Exercise 3.** Two-tailed, $\alpha = 0.10$: find both critical values and the verdict on 68.442. Compare with the two-tailed p-value (0.0663) and confirm the methods agree at this $\alpha$ too.

**Exercise 4.** *The p-value, redefined.* For observed 68.442, find the $\alpha$ at which the right-tailed critical value lands exactly on 68.442. What is that $\alpha$? Formulate the general principle ("the p-value is the smallest $\alpha$ such that...").

### Advanced

**Exercise 5.** *The advance rule's payoff.* Using the fixed rule "reject iff $\bar{x} > 68.26$", compute the Type II error probability if the true mean is actually $\mu = 68.5$ — i.e. $P(\bar{X} \le 68.26 \mid \mu = 68.5)$ — and verify by simulation. Why couldn't this number even be *defined* without a data-independent decision rule?

**Exercise 6.** *Why tables speak z.* Convert the height test's critical value to the z scale and show it is $z_{0.95} = 1.645$ — independent of $\mu_0$, $\sigma$, and $n$. Then verify: three different testing problems ($\mu_0 = 100, \sigma = 15, n = 25$; $\mu_0 = 0.5, \sigma = 0.1, n = 100$; the height test) all share the same z-scale critical value, each with its own $\bar{x}$-scale threshold.

## 7. Solutions

In [ ]:
# Exercise 1: k at alpha = 0.10
k10 = critical_values(0.10, "right")
print(f"k_0.10 = 66.7 + {z_quantile(0.90):.3f} * {SE:.4f} = {k10:.2f}")
print(f"68.442 > {k10:.2f} -> reject at alpha = 0.10")
for a in (0.10, 0.05, 0.01):
    print(f"  k_{a} = {critical_values(a, 'right'):.2f}")
print("Smaller alpha -> larger k: stricter tests demand more extreme evidence.")

In [ ]:
# Exercise 2: left-tailed at alpha = 0.01
k_l01 = critical_values(0.01, "left")
print(f"k_0.01 (left) = 66.7 - {z_quantile(0.99):.3f} * {SE:.4f} = {k_l01:.2f}")
print(f"rule: reject if x-bar < {k_l01:.2f}")
print(f"64.252 < {k_l01:.2f}? {64.252 < k_l01} -> reject at alpha = 0.01")
print("Consistent: its p-value 0.0049 < 0.01, so both methods reject.")

In [ ]:
# Exercise 3: two-tailed at alpha = 0.10
lo10, hi10 = critical_values(0.10, "two")
print(f"critical pair: [{lo10:.2f}, {hi10:.2f}]  (quantiles 0.05 and 0.95)")
outside = not (lo10 <= 68.442 <= hi10)
print(f"68.442 beyond the upper critical value {hi10:.2f}? {outside} -> "
      f"{'reject' if outside else 'do not reject'} H0")
print("p-value check: 0.0663 < 0.10 -> reject as well. The methods agree at")
print("this alpha too -- note the same data that survived the two-tailed test")
print("at alpha = 0.05 falls at alpha = 0.10.")

In [ ]:
# Exercise 4: the alpha that lands k exactly on the observation
# P(X-bar > 68.442 | H0) is by definition the alpha whose quantile is 68.442
alpha_star = 1 - Phi((68.442 - MU0) / SE)
k_check = critical_values(alpha_star, "right")
print(f"alpha* = {alpha_star:.4f}, and k_alpha* = {k_check:.3f} (= the observation)")
print("\nPrinciple: the p-value IS the smallest alpha at which the observed")
print("statistic still rejects -- shrink alpha below it and the critical value")
print("slides past your observation.")

In [ ]:
# Exercise 5: Type II probability of the advance rule
k = 68.26
mu_alt = 68.5
beta = Phi((k - mu_alt) / SE)
print(f"beta = P(X-bar <= {k} | mu = {mu_alt}) = {beta:.3f}")

sim = rng.normal(mu_alt, SE, size=200_000)
print(f"simulated: {(sim <= k).mean():.3f}")
print("\nWithout a fixed rule there is no fixed event 'fail to reject' to price:")
print("the p-value method's threshold moves with each dataset, but the advance")
print("rule freezes the rejection region, making beta a plain probability --")
print("the door to the next video.")

In [ ]:
# Exercise 6: the universal z-scale critical value
problems = [("heights", 66.7, 3.0, 10),
            ("IQ scores", 100.0, 15.0, 25),
            ("conversion rate", 0.5, 0.1, 100)]
z_crit = z_quantile(0.95)
print(f"z-scale critical value for every right-tailed alpha = 0.05 test: {z_crit:.3f}\n")
for name, mu0, sig, n in problems:
    k_x = mu0 + z_crit * sig / np.sqrt(n)
    print(f"{name:>15s}: reject if x-bar > {k_x:.3f}   (or equivalently z > {z_crit:.3f})")
print("\nOne z threshold serves every problem -- which is why printed tables and")
print("software quote critical values on the z scale.")

## 8. Conclusion

- The **critical value $k_\alpha$** is the least extreme observation that still rejects — the sample whose p-value equals $\alpha$ exactly; anything more extreme has $p \le \alpha$ automatically.
- Computed as **quantiles of the null distribution**: $1-\alpha$ (right-tailed), $\alpha$ (left-tailed), the $\alpha/2$ pair (two-tailed) — with decision rules "beyond the threshold → reject."
- The video's numbers: $k_{0.05} = 68.26$ (68.442 rejects), $k_{0.01} = 68.91$ (it doesn't) — smaller $\alpha$, farther threshold, more evidence demanded.
- The **p-value and critical-value methods always agree**; the p-value is precisely the smallest $\alpha$ at which the observation would reject.
- The advance rule's edge: it exists before the data, built from design conditions alone — which freezes the rejection region and makes **Type II error probabilities** computable. That computation is the next video.